In [0]:
# Data manipulation
import pandas as pd
import numpy as np

In [0]:
file_path = "file:/Workspace/Users/amansolanki2613@gmail.com/groww/data/users.csv"
display(
    dbutils.fs.ls(
        "file:/Workspace/Users/amansolanki2613@gmail.com/groww/data/"
    )
)

In [0]:
base_path = "file:/Workspace/Users/amansolanki2613@gmail.com/groww/data/"

users_df = spark.read.csv(base_path + "users.csv", header=True, inferSchema=True)

stocks_df = spark.read.csv(base_path + "stocks_orders.csv", header=True, inferSchema=True)

fno_df = spark.read.csv(base_path + "fno_orders.csv", header=True, inferSchema=True)

mf_df = spark.read.csv(base_path + "mf_orders.csv", header=True, inferSchema=True)

ipo_df = spark.read.csv(base_path + "ipo_orders.csv", header=True, inferSchema=True)

commodity_df = spark.read.csv(base_path + "commodity_orders.csv", header=True, inferSchema=True)

In [0]:
# Create persistent Delta tables

users_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("users")

stocks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("stocks_orders")

fno_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fno_orders")

mf_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mf_orders")

ipo_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ipo_orders")

commodity_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("commodity_orders")

In [0]:
print(users_df)
print(stocks_df)
print(fno_df)
print(mf_df)
print(ipo_df)
print(commodity_df)

In [0]:
dataframes = {
    "users": users_df,
    "stocks": stocks_df,
    "fno": fno_df,
    "mf": mf_df,
    "ipo": ipo_df,
    "commodity": commodity_df
}

for name, df in dataframes.items():

    print(f"\n{'='*50}")
    print(f"DATASET: {name}")

    print("Rows:", df.count())
    print("Columns:", len(df.columns))

    df.printSchema()

In [0]:
def standardize_text(df, columns):

    for c in columns:
        df = df.withColumn(
            c,
            F.when(
                F.col(c).isNotNull(),
                F.upper(F.trim(F.col(c).cast("string")))
            ).otherwise(None)
        )

    return df

In [0]:
from pyspark.sql import functions as F
for name, df in dataframes.items():

    if "order_status" in df.columns:

        print(f"\n{name.upper()} STATUS DISTRIBUTION")

        display(
            df.groupBy("order_status")
              .count()
              .orderBy(F.desc("count"))
        )

STOCK TABLE ANALYSIS

TABLE CREATED FOR STOCK ORDERS which are SUCCESSFUL only

In [0]:
%sql
CREATE OR REPLACE TABLE
stocks_successful AS

SELECT
    third_party_id,
    groww_order_id,
    created_at,
    buy_sell,
    product,
    symbol_name,
    filled_qty,
    turnover_inr,
    original_brokerage,
    charges,
    order_platform,
    is_sip_order,
    is_etf_order,
    order_status

FROM stocks_orders

WHERE order_status = 'SUCCESS';

In [0]:
%sql
CREATE OR REPLACE TABLE
stocks_user_metrics AS

SELECT

    third_party_id,

    -- Monetary metrics
    SUM(COALESCE(original_brokerage, 0))
        AS total_brokerage,

    SUM(COALESCE(turnover_inr, 0))
        AS total_turnover,

    SUM(COALESCE(charges, 0))
        AS total_charges,

    -- Trading activity
    COUNT(DISTINCT groww_order_id)
        AS total_successful_orders,

    SUM(COALESCE(filled_qty, 0))
        AS total_shares_traded,

    COUNT(DISTINCT TO_DATE(created_at))
        AS active_trading_days,

    -- Trading behaviour
    COUNT(DISTINCT product)
        AS distinct_trading_products,

    COUNT(DISTINCT symbol_name)
        AS distinct_stocks_traded,

    COUNT(DISTINCT order_platform)
        AS platforms_used,

    SUM(CASE WHEN buy_sell = 'B'
        THEN 1 ELSE 0 END) AS buy_orders,

    SUM(CASE WHEN buy_sell = 'S'
        THEN 1 ELSE 0 END) AS sell_orders,

    SUM(CASE WHEN is_sip_order = 1
        THEN 1 ELSE 0 END) AS sip_orders,

    SUM(CASE WHEN is_etf_order = 1
        THEN 1 ELSE 0 END) AS etf_orders,

    MAX(created_at) AS last_order_date
    

FROM stocks_successful

GROUP BY third_party_id;

In [0]:
%sql
SELECT
    third_party_id,
    total_brokerage,
    total_turnover,
    total_successful_orders,
    active_trading_days,
    distinct_trading_products,

    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment
    

FROM stocks_user_metrics
ORDER BY total_brokerage desc ;

In [0]:
%sql
CREATE OR REPLACE TABLE stocks_user_segments AS

SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment

FROM stocks_user_metrics;

In [0]:
%sql
SELECT
    value_segment,
    COUNT(*) AS total_users,
    ROUND(AVG(total_brokerage), 2) AS avg_brokerage,
    ROUND(SUM(total_brokerage), 2) AS total_brokerage,

    ROUND(
        100.0 * SUM(total_brokerage) /
        SUM(SUM(total_brokerage)) OVER (),
        2
    ) AS revenue_contribution_pct

FROM stocks_user_segments

GROUP BY value_segment

ORDER BY value_segment;